In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import numpy as np
import random

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Seed fixed.")

Seed fixed.


In [3]:
IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

In [4]:
base_dataset = torchvision.datasets.ImageFolder(
    root="data/mri-data/full_dataset"
)

X = np.arange(len(base_dataset))
y = [label for _, label in base_dataset.samples]

print("Total samples:", len(base_dataset))
print("Classes:", base_dataset.classes)
print(len(base_dataset))
print(len(set([path for path, _ in base_dataset.samples])))

Total samples: 6395
Classes: ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']
6395
6395


In [5]:
class_weights = torch.tensor([1.5, 3.0, 1.0, 1.2], dtype=torch.float)

In [6]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_results = []

In [7]:
device = torch.device("cuda")
print("Using device:", device)

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n========== FOLD {fold+1} ==========")

    # Fresh dataset instances
    train_dataset = torchvision.datasets.ImageFolder(
        root="data/mri-data/full_dataset",
        transform=train_transform
    )

    val_dataset = torchvision.datasets.ImageFolder(
        root="data/mri-data/full_dataset",
        transform=val_transform
    )

    train_subset = Subset(train_dataset, train_idx)
    val_subset = Subset(val_dataset, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)

    # ----- Model -----
    model = models.resnet18(weights="IMAGENET1K_V1")
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 4)
    model = model.to(device)

    for param in model.parameters():
        param.requires_grad = False

    for param in model.layer4.parameters():
        param.requires_grad = True

    for param in model.fc.parameters():
        param.requires_grad = True

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = optim.Adam(
        list(model.layer4.parameters()) + list(model.fc.parameters()),
        lr=0.0001
    )

    EPOCHS = 6

    # ----- Training -----
    for epoch in range(EPOCHS):
        model.train()
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # ----- Validation -----
    model.eval()
    preds = []
    true = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            preds.extend(predicted.cpu().numpy())
            true.extend(labels.cpu().numpy())

    acc = accuracy_score(true, preds)
    fold_results.append(acc)

    print("Fold Accuracy:", acc)

print("\n===== CROSS VALIDATION RESULTS =====")
print("Fold Accuracies:", fold_results)
print("Mean Accuracy:", np.mean(fold_results))
print("Std Deviation:", np.std(fold_results))

Using device: cuda

========== FOLD 1 ==========
Fold Accuracy: 0.8606941838649156

========== FOLD 2 ==========
Fold Accuracy: 0.8348968105065666

========== FOLD 3 ==========
Fold Accuracy: 0.8657907085875176

===== CROSS VALIDATION RESULTS =====
Fold Accuracies: [0.8606941838649156, 0.8348968105065666, 0.8657907085875176]
Mean Accuracy: 0.8537939009863332
Std Deviation: 0.013523280236007563


In [3]:
torch.save(model.state_dict(), "best_model.pth")
print("Model saved successfully")

NameError: name 'model' is not defined